# Research Results — CI/CD Test Charts

Comparing 3 Prompting Strategies: **BP** (Basic Prompting), **CE** (Context Engineering), **SDD** (Specification-Driven Development)

Across 3 Features: **IM** (Inventory Management), **SC** (Shopping Cart), **PD** (Promotions & Discounts)

With 4 Testing Dimensions: **Unit/Integration Tests**, **CodeQL (SAST)**, **DAST (ZAP)**, **SonarQube**

> Based on 9 analysis versions (BP×3 + CE×3 + SDD×3) — `SCSD01_v2` excluded as reproducibility-only.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import pandas as pd
import numpy as np
from textwrap import wrap

# Readability-first chart style: keep labels clear in exported PNGs and notebooks.
sns.set_theme(style='whitegrid', font_scale=1.15)
matplotlib.rcParams['figure.dpi'] = 150
matplotlib.rcParams['savefig.dpi'] = 300
matplotlib.rcParams['figure.facecolor'] = 'white'
matplotlib.rcParams['axes.titlesize'] = 19
matplotlib.rcParams['axes.labelsize'] = 14
matplotlib.rcParams['axes.labelweight'] = 'semibold'
matplotlib.rcParams['xtick.labelsize'] = 12
matplotlib.rcParams['ytick.labelsize'] = 12
matplotlib.rcParams['legend.fontsize'] = 12
matplotlib.rcParams['font.weight'] = 'normal'
matplotlib.rcParams['figure.autolayout'] = False
SAVE_KWARGS = {'bbox_inches': 'tight', 'pad_inches': 0.12}


def bold_axis_text(ax, tick_weight='normal'):
    ax.xaxis.label.set_fontweight('semibold')
    ax.yaxis.label.set_fontweight('semibold')
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight(tick_weight)


def soften_axis(ax):
    ax.grid(axis='y', alpha=0.25, linewidth=0.8)
    ax.grid(axis='x', visible=False)
    sns.despine(ax=ax, left=False, bottom=False)
    ax.tick_params(axis='both', which='major', pad=4)


def wrap_labels(labels, width=18):
    return ['\n'.join(wrap(str(label), width=width, break_long_words=False)) for label in labels]


def version_group_guides(ax, y, fontsize=14):
    ax.axvspan(-0.5, 2.5, alpha=0.035, color=STRATEGY_COLORS['BP'])
    ax.axvspan(2.5, 5.5, alpha=0.035, color=STRATEGY_COLORS['CE'])
    ax.axvspan(5.5, 8.5, alpha=0.035, color=STRATEGY_COLORS['SDD'])
    ax.text(1, y, 'BP', ha='center', fontsize=fontsize, fontstyle='italic', alpha=0.45)
    ax.text(4, y, 'CE', ha='center', fontsize=fontsize, fontstyle='italic', alpha=0.45)
    ax.text(7, y, 'SDD', ha='center', fontsize=fontsize, fontstyle='italic', alpha=0.45)


# สี palette สำหรับ 3 strategies
STRATEGY_COLORS = {
    'BP': '#e74c3c',   # แดง
    'CE': '#3498db',   # น้ำเงิน
    'SDD': '#2ecc71',  # เขียว
}
STRATEGY_NAMES = {
    'BP': 'Basic Prompting',
    'CE': 'Context Engineering',
    'SDD': 'Spec-Driven Dev',
}

print('Libraries loaded successfully!')

## Data Preparation

In [ ]:
# === 9 Versions: BP×3, CE×3, SDD×3 (SCSD01_v2 excluded — reproducibility test only) ===
# Order: IM(BP,CE,SD), SC(BP,CE,SD), PD(BP,CE,SD)
VERSIONS = ['IMBP01','IMCE01','IMSD01',
            'SCBP01','SCCE01','SCSD01',
            'PDBP01','PDCE01','PDSD01']
STRATEGIES = ['BP','CE','SDD','BP','CE','SDD','BP','CE','SDD']
FEATURES   = ['IM','IM','IM','SC','SC','SC','PD','PD','PD']
TOOLS = ['ChatGPT','ChatGPT','Codex',
         'ChatGPT','ChatGPT','Codex',
         'ChatGPT','ChatGPT','Codex']

# === Unit Test Data ===
df_tests = pd.DataFrame({
    'Version':  VERSIONS,
    'Strategy': STRATEGIES,
    'Feature':  FEATURES,
    'Tool':     TOOLS,
    'Passed': [7,7,17, 5,5,16, 6,6,18],
    'Total':  [7,7,17, 5,5,16, 6,6,18],
})
df_tests['Pass_Rate'] = df_tests['Passed'] / df_tests['Total'] * 100

# === CodeQL Data ===
df_codeql = pd.DataFrame({
    'Version':  VERSIONS,
    'Strategy': STRATEGIES,
    'Feature':  FEATURES,
    'Tool':     TOOLS,
    'High':   [5,3,0, 0,6,0, 0,0,0],
    'Medium': [0,1,0, 0,0,0, 0,1,0],
})
df_codeql['Total_Alerts'] = df_codeql['High'] + df_codeql['Medium']

# === DAST Data (9 versions) ===
VERSIONS_DAST    = ['IMBP01','IMCE01','IMSD01',
                    'SCBP01','SCCE01','SCSD01',
                    'PDBP01','PDCE01','PDSD01']
STRATEGIES_DAST  = ['BP','CE','SDD','BP','CE','SDD','BP','CE','SDD']
FEATURES_DAST    = ['IM','IM','IM','SC','SC','SC','PD','PD','PD']
df_dast = pd.DataFrame({
    'Version':     VERSIONS_DAST,
    'Strategy':    STRATEGIES_DAST,
    'Feature':     FEATURES_DAST,
    'FAIL': [0,0,0, 0,0,0, 0,0,0],
    'WARN': [7,7,7, 7,8,8, 8,7,9],
    'PASS': [60,60,60, 60,59,59, 59,60,58],
    'Server_Leak': [0,0,0, 0,1,0, 1,0,1],
})

# === SonarQube Data ===
df_sonar = pd.DataFrame({
    'Version':  VERSIONS,
    'Strategy': STRATEGIES,
    'Feature':  FEATURES,
    'Tool':     TOOLS,
    'Security':        [0,0,0, 0,0,1, 0,0,1],
    'Reliability':     [3,19,12, 9,9,0, 1,16,6],
    'Maintainability': [7,26,15, 18,12,5, 10,20,9],
    'Duplications':    [6.90,0.00,3.30, 5.40,4.30,26.90, 1.40,0.00,0.00],
})

print('Data prepared!')
print(f'Versions: {len(VERSIONS)}, Strategies: 3 (BP=3, CE=3, SDD=3) — SCSD01_v2 excluded (reproducibility test only)')
print(f'  Unit Tests: {df_tests["Passed"].sum()}/{df_tests["Total"].sum()} passed')
print(f'  CodeQL: {df_codeql["Total_Alerts"].sum()} total alerts')
print(f'  DAST: {df_dast["FAIL"].sum()} fails, {df_dast["WARN"].sum()} warns ({len(VERSIONS_DAST)} versions)')
print(f'  SonarQube: Rel={df_sonar["Reliability"].sum()}, Maint={df_sonar["Maintainability"].sum()}')

---
## Chart 1: CodeQL Security Alerts by Strategy (Bar Chart)
Key finding ของงานวิจัย — ความแตกต่างด้าน security alerts ระหว่าง strategy

In [ ]:
fig, ax = plt.subplots(figsize=(8.8, 5.6))

strategies = ['BP', 'CE', 'SDD']
high_by_strategy = df_codeql.groupby('Strategy')['High'].sum().reindex(strategies)
med_by_strategy = df_codeql.groupby('Strategy')['Medium'].sum().reindex(strategies)

x = np.arange(len(strategies))
width = 0.42

bars_high = ax.bar(x, high_by_strategy, width, label='High Severity', color='#e74c3c', edgecolor='white')
bars_med = ax.bar(x, med_by_strategy, width, bottom=high_by_strategy, label='Medium Severity', color='#f39c12', edgecolor='white')

for i, (h, m) in enumerate(zip(high_by_strategy, med_by_strategy)):
    total = h + m
    if total > 0:
        ax.text(i, total + 0.35, f'{int(total)} total', ha='center', va='bottom',
                fontweight='semibold', fontsize=15)
        if h >= 2:
            ax.text(i, h / 2, f'{int(h)}', ha='center', va='center',
                    fontweight='semibold', fontsize=15, color='white')
        if m >= 1.5:
            ax.text(i, h + m / 2, f'{int(m)}', ha='center', va='center',
                    fontweight='semibold', fontsize=14, color='white')
    else:
        ax.text(i, 0.35, '0', ha='center', va='bottom', fontweight='semibold', fontsize=15, color='#2ecc71')

ax.set_xlabel('Prompting Strategy')
ax.set_ylabel('Number of Security Alerts')
ax.set_title('CodeQL (SAST)', fontweight='semibold', pad=14)
ax.set_xticks(x)
ax.set_xticklabels([f'{s}\n{STRATEGY_NAMES[s]}' for s in strategies], fontsize=14)
ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), frameon=True, borderaxespad=0)
ax.set_ylim(0, 16)
ax.yaxis.set_major_locator(plt.MultipleLocator(2))

soften_axis(ax)
bold_axis_text(ax)
plt.tight_layout(pad=0.8)
plt.savefig('chart1_codeql_by_strategy.png', **SAVE_KWARGS)
plt.show()
print('Saved: chart1_codeql_by_strategy.png')

---
## Chart 2: CodeQL Alerts by Version (Grouped Bar Chart)
แสดงรายละเอียดว่า version ไหนมี alert อะไรบ้าง

In [ ]:
fig, ax = plt.subplots(figsize=(12.8, 5.8))

versions_order = ['IMBP01','SCBP01','PDBP01','IMCE01','SCCE01','PDCE01','IMSD01','SCSD01','PDSD01']
df_codeql_ordered = df_codeql.set_index('Version').reindex(versions_order).reset_index()
versions = df_codeql_ordered['Version'].tolist()
x = np.arange(len(versions))
width = 0.28

high_vals = df_codeql_ordered['High']
med_vals = df_codeql_ordered['Medium']

bars_high = ax.bar(x - width/2, high_vals, width, label='High Severity', color='#e74c3c', edgecolor='white')
bars_med = ax.bar(x + width/2, med_vals, width, label='Medium Severity', color='#f39c12', edgecolor='white')

for i, (h, m) in enumerate(zip(high_vals, med_vals)):
    total = h + m
    if h > 0:
        ax.text(x[i] - width/2, h + 0.12, f'{int(h)}', ha='center', va='bottom', fontweight='semibold', fontsize=14)
    if m > 0:
        ax.text(x[i] + width/2, m + 0.12, f'{int(m)}', ha='center', va='bottom', fontweight='semibold', fontsize=14)
    if total == 0:
        ax.text(x[i], 0.18, '0', ha='center', va='bottom', fontweight='semibold', fontsize=14, color='#2ecc71')

version_group_guides(ax, y=7.15, fontsize=14)

ax.set_xlabel('Version')
ax.set_ylabel('Number of Security Alerts')
ax.set_title('CodeQL (SAST) Alerts', fontweight='semibold', pad=14)
ax.set_xticks(x)
ax.set_xticklabels(versions, rotation=0, ha='center', fontsize=14)
ax.set_ylim(0, 8)
ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), frameon=True, borderaxespad=0)
ax.yaxis.set_major_locator(plt.MultipleLocator(1))

soften_axis(ax)
bold_axis_text(ax)
plt.tight_layout(pad=0.8)
plt.savefig('chart2_codeql_by_version.png', **SAVE_KWARGS)
plt.show()
print('Saved: chart2_codeql_by_version.png')

---
## Chart 3: DAST — FAIL / WARN / PASS by Version (Stacked Bar)

In [ ]:
fig, ax = plt.subplots(figsize=(12.8, 5.8))

versions_order = ['IMBP01','SCBP01','PDBP01','IMCE01','SCCE01','PDCE01','IMSD01','SCSD01','PDSD01']
df_dast_ordered = df_dast.set_index('Version').reindex(versions_order).reset_index()
versions = df_dast_ordered['Version'].tolist()
x = np.arange(len(versions))
width = 0.48

ax.bar(x, df_dast_ordered['FAIL'], width, label='FAIL', color='#e74c3c', edgecolor='white')
ax.bar(x, df_dast_ordered['WARN'], width, bottom=df_dast_ordered['FAIL'], label='WARN', color='#f39c12', edgecolor='white')
ax.bar(x, df_dast_ordered['PASS'], width, bottom=df_dast_ordered['FAIL']+df_dast_ordered['WARN'], label='PASS', color='#2ecc71', edgecolor='white')

for i, row in df_dast_ordered.iterrows():
    ax.text(i, row['FAIL'] + row['WARN']/2, str(int(row['WARN'])),
            ha='center', va='center', fontweight='semibold', fontsize=15, color='white')
    ax.text(i, row['FAIL'] + row['WARN'] + row['PASS']/2, str(int(row['PASS'])),
            ha='center', va='center', fontweight='semibold', fontsize=15, color='white')

version_group_guides(ax, y=69, fontsize=14)

ax.set_xlabel('Version')
ax.set_ylabel('Number of Rules')
ax.set_title('DAST (ZAP) - FAIL / WARN / PASS', fontweight='semibold', pad=14)
ax.set_xticks(x)
ax.set_xticklabels(versions, rotation=0, ha='center', fontsize=14)
ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), frameon=True, borderaxespad=0)
ax.set_ylim(0, 72)

soften_axis(ax)
bold_axis_text(ax)
plt.tight_layout(pad=0.8)
plt.savefig('chart3_dast_by_version.png', **SAVE_KWARGS)
plt.show()
print('Saved: chart3_dast_by_version.png')

---
## Chart 4: DAST — Average WARN / PASS & Server Leak by Strategy

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.4, 5.2), constrained_layout=True)
strategies = ['BP', 'CE', 'SDD']
colors = [STRATEGY_COLORS[s] for s in strategies]

# Left: Average Warnings
ax = axes[0]
avg_warn = df_dast.groupby('Strategy')['WARN'].mean().reindex(strategies)
bars = ax.bar(strategies, avg_warn, color=colors, edgecolor='white', width=0.42)
for bar, val in zip(bars, avg_warn):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.12, f'{val:.1f}',
            ha='center', va='bottom', fontweight='semibold', fontsize=14)
ax.set_ylabel('Average Warnings')
ax.set_title('Avg WARN', fontsize=15, fontweight='semibold', pad=8)
ax.set_ylim(0, 10)

# Middle: Average Pass
ax = axes[1]
avg_pass = df_dast.groupby('Strategy')['PASS'].mean().reindex(strategies)
bars = ax.bar(strategies, avg_pass, color=colors, edgecolor='white', width=0.42)
for bar, val in zip(bars, avg_pass):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.08, f'{val:.1f}',
            ha='center', va='bottom', fontweight='semibold', fontsize=14)
ax.set_ylabel('Average Pass')
ax.set_title('Avg PASS', fontsize=15, fontweight='semibold', pad=8)
ax.set_ylim(55, 62)

# Right: Server Leak count
ax = axes[2]
leak_count = df_dast.groupby('Strategy')['Server_Leak'].sum().reindex(strategies)
bars = ax.bar(strategies, leak_count, color=colors, edgecolor='white', width=0.42)
for bar, val in zip(bars, leak_count):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.08, f'{int(val)}',
            ha='center', va='bottom', fontweight='semibold', fontsize=14)
ax.set_ylabel('Server Leak Count')
ax.set_title('Server Version Leak', fontsize=15, fontweight='semibold', pad=8)
ax.set_ylim(0, 4)
ax.yaxis.set_major_locator(plt.MultipleLocator(1))

for a in axes.flat:
    a.set_xlabel('Strategy')
    soften_axis(a)
    bold_axis_text(a)

fig.suptitle('DAST (ZAP) Security Scan Results by Strategy', fontsize=18, fontweight='semibold')
plt.savefig('chart4_dast_by_strategy.png', **SAVE_KWARGS)
plt.show()
print('Saved: chart4_dast_by_strategy.png')

---
## Chart 5: DAST Heatmap — Warning Types by Version

In [ ]:
# ZAP warning types per version (1 = present, 0 = not present)
# Aligned with CI_TEST_RESULTS.md and GitHub Actions run #24299653751
warnings = [
    'Anti-clickjacking Header Missing',
    'CSP Header Not Set',
    'COEP Header Missing',
    'COOP Header Missing',
    'CORP Header Missing',
    'Permissions Policy Not Set',
    'X-Content-Type-Options Header Missing',
    'Non-Cacheable Content',
    'Modern Web Application',
    'Server Version Leak',
    'In-Page Banner Info Leak',
    'Subresource Integrity Missing',
]

warning_matrix = {
    'IMBP01': [1,1,1,1,1,1,1,1,1,0,0,0],
    'IMCE01': [1,1,1,1,1,1,1,1,1,0,0,0],
    'IMSD01': [1,1,1,1,1,1,1,1,1,0,0,0],
    'SCBP01': [1,1,1,1,1,1,1,1,1,0,0,0],
    'SCCE01': [1,1,1,1,1,1,1,1,1,1,0,0],
    'SCSD01': [1,1,1,1,1,1,1,1,1,0,0,1],
    'PDBP01': [1,1,1,1,1,1,1,1,1,1,0,0],
    'PDCE01': [1,1,1,1,1,1,1,1,1,0,0,0],
    'PDSD01': [1,1,1,1,1,1,1,1,1,1,1,0],
}

versions = ['IMBP01','SCBP01','PDBP01','IMCE01','SCCE01','PDCE01','IMSD01','SCSD01','PDSD01']
heatmap_data = [warning_matrix[v] for v in versions]
df_heatmap = pd.DataFrame(heatmap_data, index=versions, columns=warnings)

fig, ax = plt.subplots(figsize=(12.6, 7.2))
sns.heatmap(df_heatmap.T, annot=True, fmt='d', cmap=['#2ecc71','#e74c3c'],
            linewidths=0.8, linecolor='white', cbar=False, ax=ax,
            annot_kws={'fontsize': 14, 'fontweight': 'semibold'})

ax.set_title('DAST (ZAP) Warning Types\nGreen = Not Present (0), Red = Present (1)',
             fontweight='semibold', pad=16)
ax.set_xlabel('Version')
ax.set_ylabel('')
ax.set_xticklabels(versions, rotation=0, ha='center', fontsize=13)
ax.set_yticklabels(wrap_labels(warnings, width=40), rotation=0, fontsize=13)

bold_axis_text(ax)
plt.tight_layout(pad=0.8)
fig.subplots_adjust(left=0.31, right=0.98)
plt.savefig('chart5_dast_heatmap.png', **SAVE_KWARGS)
plt.show()
print('Saved: chart5_dast_heatmap.png')

---
## Chart 6: SonarQube — Reliability & Maintainability Issues by Strategy

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13.2, 9.8), constrained_layout=True)
strategies = ['BP', 'CE', 'SDD']
colors = [STRATEGY_COLORS[s] for s in strategies]

# Top-Left: Security
ax = axes[0][0]
avg_sec = df_sonar.groupby('Strategy')['Security'].mean().reindex(strategies)
bars = ax.bar(strategies, avg_sec, color=colors, edgecolor='white', width=0.46)
for bar, val in zip(bars, avg_sec):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.025, f'{val:.2f}',
            ha='center', va='bottom', fontweight='semibold', fontsize=18)
ax.set_ylabel('Avg Open Issues', fontsize=16)
ax.set_title('Security Issues', fontsize=19, fontweight='semibold', pad=10)
ax.set_ylim(0, max(max(avg_sec) * 1.65, 0.6))

# Top-Right: Reliability
ax = axes[0][1]
avg_rel = df_sonar.groupby('Strategy')['Reliability'].mean().reindex(strategies)
bars = ax.bar(strategies, avg_rel, color=colors, edgecolor='white', width=0.46)
for bar, val in zip(bars, avg_rel):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.45, f'{val:.1f}',
            ha='center', va='bottom', fontweight='semibold', fontsize=18)
ax.set_ylabel('Avg Open Issues', fontsize=16)
ax.set_title('Reliability Issues', fontsize=19, fontweight='semibold', pad=10)
ax.set_ylim(0, max(avg_rel) * 1.42)

# Bottom-Left: Maintainability
ax = axes[1][0]
avg_maint = df_sonar.groupby('Strategy')['Maintainability'].mean().reindex(strategies)
bars = ax.bar(strategies, avg_maint, color=colors, edgecolor='white', width=0.46)
for bar, val in zip(bars, avg_maint):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.45, f'{val:.1f}',
            ha='center', va='bottom', fontweight='semibold', fontsize=18)
ax.set_ylabel('Avg Open Issues', fontsize=16)
ax.set_title('Maintainability Issues', fontsize=19, fontweight='semibold', pad=10)
ax.set_ylim(0, max(avg_maint) * 1.42)

# Bottom-Right: Duplications
ax = axes[1][1]
avg_dup = df_sonar.groupby('Strategy')['Duplications'].mean().reindex(strategies)
bars = ax.bar(strategies, avg_dup, color=colors, edgecolor='white', width=0.46)
for bar, val in zip(bars, avg_dup):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.25, f'{val:.2f}%',
            ha='center', va='bottom', fontweight='semibold', fontsize=18)
ax.set_ylabel('Avg Duplication %', fontsize=16)
ax.set_title('Code Duplications', fontsize=19, fontweight='semibold', pad=10)
ax.set_ylim(0, max(avg_dup) * 1.65)

for row in axes:
    for a in row:
        a.set_xlabel('Strategy', fontsize=16)
        a.tick_params(axis='both', labelsize=15, pad=5)
        soften_axis(a)
        bold_axis_text(a)

fig.suptitle('SonarQube - All 4 Dimensions by Strategy (Lower = Better)', fontsize=22, fontweight='semibold')
plt.savefig('chart6_sonarqube_by_strategy.png', **SAVE_KWARGS)
plt.show()
print('Saved: chart6_sonarqube_by_strategy.png')


---
## Chart 7: SonarQube — Issues Heatmap by Version

In [ ]:
fig, ax = plt.subplots(figsize=(13.0, 6.1))

versions_by_strategy = ['IMBP01','SCBP01','PDBP01',
                         'IMCE01','SCCE01','PDCE01',
                         'IMSD01','SCSD01','PDSD01']

df_sonar_indexed = df_sonar.set_index('Version')
sonar_heatmap = df_sonar_indexed.loc[versions_by_strategy, ['Security','Reliability','Maintainability']].copy()
sonar_heatmap['Duplications (%)'] = df_sonar_indexed.loc[versions_by_strategy, 'Duplications']

annot_data = sonar_heatmap.copy()
annot_text = annot_data.astype(str)
for v in versions_by_strategy:
    for col in ['Security','Reliability','Maintainability']:
        annot_text.loc[v, col] = str(int(annot_data.loc[v, col]))
    annot_text.loc[v, 'Duplications (%)'] = f"{annot_data.loc[v, 'Duplications (%)']:.1f}%"

sns.heatmap(sonar_heatmap.T, annot=False, fmt='', cmap='YlOrRd',
            linewidths=1, linecolor='white', ax=ax,
            cbar_kws={'label': 'Open Issues / %', 'shrink': 0.82})

rows = ['Security', 'Reliability', 'Maintainability', 'Duplications (%)']
for row_i, row_name in enumerate(rows):
    for col_i, version in enumerate(versions_by_strategy):
        text = annot_text.loc[version, row_name]
        val = sonar_heatmap.loc[version, row_name]
        max_val = sonar_heatmap.values.max()
        color = 'white' if val > max_val * 0.6 else 'black'
        ax.text(col_i + 0.5, row_i + 0.5, text,
                ha='center', va='center',
                fontsize=14, fontweight='semibold', color=color)

ax.axvline(x=3, color='black', linewidth=1.8, linestyle='--', alpha=0.55)
ax.axvline(x=6, color='black', linewidth=1.8, linestyle='--', alpha=0.55)

ax.text(1.5, -0.22, 'BP', ha='center', va='top', fontsize=14, fontweight='semibold',
        color=STRATEGY_COLORS['BP'], transform=ax.get_xaxis_transform())
ax.text(4.5, -0.22, 'CE', ha='center', va='top', fontsize=14, fontweight='semibold',
        color=STRATEGY_COLORS['CE'], transform=ax.get_xaxis_transform())
ax.text(7.5, -0.22, 'SDD', ha='center', va='top', fontsize=14, fontweight='semibold',
        color=STRATEGY_COLORS['SDD'], transform=ax.get_xaxis_transform())

ax.set_title('SonarQube - All 4 Dimensions by Version', fontweight='semibold', pad=14)
ax.set_xlabel('Version')
ax.set_ylabel('')
ax.set_xticklabels(versions_by_strategy, rotation=0, ha='center', fontsize=14)
ax.set_yticklabels(rows, rotation=0, fontsize=14)

bold_axis_text(ax)
plt.tight_layout(pad=0.8)
plt.savefig('chart7_sonarqube_heatmap.png', **SAVE_KWARGS)
plt.show()
print('Saved: chart7_sonarqube_heatmap.png')

---
## Chart 8: SonarQube — Code Duplications by Version

In [ ]:
fig, ax = plt.subplots(figsize=(12.4, 5.6))

versions_order = ['IMBP01','SCBP01','PDBP01','IMCE01','SCCE01','PDCE01','IMSD01','SCSD01','PDSD01']
df_sonar_ordered = df_sonar.set_index('Version').reindex(versions_order).reset_index()
versions = df_sonar_ordered['Version'].tolist()
x = np.arange(len(versions))
colors_by_strategy = [STRATEGY_COLORS[s] for s in df_sonar_ordered['Strategy']]

bars = ax.bar(x, df_sonar_ordered['Duplications'], width=0.45, color=colors_by_strategy, edgecolor='white')

for i, (bar, val) in enumerate(zip(bars, df_sonar_ordered['Duplications'])):
    if val > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.35, f'{val:.1f}%',
                ha='center', va='bottom', fontweight='semibold', fontsize=14)

version_group_guides(ax, y=30, fontsize=14)

ax.set_xlabel('Version')
ax.set_ylabel('Duplication %')
ax.set_title('SonarQube - Code Duplications by Version', fontweight='semibold', pad=14)
ax.set_xticks(x)
ax.set_xticklabels(versions, rotation=0, ha='center', fontsize=14)
ax.set_ylim(0, 32)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=STRATEGY_COLORS[s], label=f'{s} ({STRATEGY_NAMES[s]})')
                   for s in ['BP','CE','SDD']]
ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=14, frameon=True, borderaxespad=0)

soften_axis(ax)
bold_axis_text(ax)
plt.tight_layout(pad=0.8)
plt.savefig('chart8_sonarqube_duplications.png', **SAVE_KWARGS)
plt.show()
print('Saved: chart8_sonarqube_duplications.png')

---
## Chart 9: Unit Tests — Pass Rate Heatmap by Strategy x Feature

In [ ]:
strategies = ['BP', 'CE', 'SDD']
features = ['IM', 'SC', 'PD']

fig, ax = plt.subplots(figsize=(9.0, 4.8))

pivot_rate = df_tests.pivot_table(values='Pass_Rate', index='Strategy', columns='Feature').reindex(strategies)[features]
pivot_passed = df_tests.pivot_table(values='Passed', index='Strategy', columns='Feature', aggfunc='sum').reindex(strategies)[features]
pivot_total = df_tests.pivot_table(values='Total', index='Strategy', columns='Feature', aggfunc='sum').reindex(strategies)[features]

annot = pivot_passed.astype(int).astype(str) + '/' + pivot_total.astype(int).astype(str)

sns.heatmap(pivot_rate, annot=annot, fmt='', cmap='Greens', vmin=80, vmax=100,
            linewidths=2, linecolor='white', ax=ax, cbar_kws={'label': 'Pass Rate (%)', 'shrink': 0.82},
            annot_kws={'fontsize': 12, 'fontweight': 'semibold'})

ax.set_title('Unit & Integration Tests - All Strategies Pass 100%', fontweight='semibold', pad=14)
ax.set_ylabel('Strategy')
ax.set_xlabel('Feature')
ax.set_yticklabels([f'{s}\n{STRATEGY_NAMES[s]}' for s in strategies], rotation=0, fontsize=14)
ax.set_xticklabels(['IM\nInventory', 'SC\nShopping Cart', 'PD\nPromotions'], fontsize=14)

bold_axis_text(ax)
plt.tight_layout(pad=0.8)
plt.savefig('chart9_unit_tests_heatmap.png', **SAVE_KWARGS)
plt.show()
print('Saved: chart9_unit_tests_heatmap.png')

---
## Chart 10: Radar — Overall Strategy Comparison (All Dimensions)

In [ ]:
strategies = ['BP', 'CE', 'SDD']

radar_data = {}
for s in strategies:
    s_codeql = df_codeql[df_codeql['Strategy'] == s]
    s_dast = df_dast[df_dast['Strategy'] == s]
    s_sonar = df_sonar[df_sonar['Strategy'] == s]
    
    # 1. CodeQL (inverse: 0 alerts = 100)
    max_alerts = 16
    codeql_score = max(0, (1 - s_codeql['Total_Alerts'].sum() / max_alerts) * 100)
    
    # 2. DAST Pass Rate
    dast_pass_score = (s_dast['PASS'].mean() / 67) * 100
    
    # 3. SonarQube Reliability (inverse: fewer = better)
    max_rel = 25
    sonar_rel_score = max(0, (1 - s_sonar['Reliability'].mean() / max_rel) * 100)
    
    # 4. SonarQube Maintainability (inverse: fewer = better)
    max_maint = 30
    sonar_maint_score = max(0, (1 - s_sonar['Maintainability'].mean() / max_maint) * 100)
    
    # 5. SonarQube Duplications (inverse: lower = better)
    max_dup = 8
    sonar_dup_score = max(0, (1 - s_sonar['Duplications'].mean() / max_dup) * 100)
    
    radar_data[s] = [codeql_score, dast_pass_score, sonar_rel_score, sonar_maint_score, sonar_dup_score]

categories = ['CodeQL\nNo Alerts', 'DAST\nPass Rate',
              'Reliability', 'Maintainability', 'Low\nDuplication']
N = len(categories)

angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10.8, 9.2), subplot_kw=dict(polar=True))

for s in strategies:
    values = radar_data[s] + radar_data[s][:1]
    ax.plot(angles, values, 'o-', linewidth=2.2, label=f'{s} ({STRATEGY_NAMES[s]})',
            color=STRATEGY_COLORS[s], markersize=5)
    ax.fill(angles, values, alpha=0.08, color=STRATEGY_COLORS[s])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=14)
ax.tick_params(axis='x', pad=14)
ax.set_ylim(0, 105)
ax.set_yticks([20, 40, 60, 80, 100])
ax.set_yticklabels(['20', '40', '60', '80', '100'], fontsize=15)
ax.set_title('Overall Strategy Comparison\n(Higher = Better)', fontweight='semibold', pad=24)
ax.legend(loc='upper left', bbox_to_anchor=(1.03, 1.02), fontsize=14, frameon=True, borderaxespad=0)

bold_axis_text(ax)
fig.subplots_adjust(left=0.14, right=0.74, top=0.86, bottom=0.12)
plt.savefig('chart10_radar_comparison.png', **SAVE_KWARGS)
plt.show()
print('Saved: chart10_radar_comparison.png')

---
## Summary

| Chart | File | Description |
|-------|------|-------------|
| 1 | `chart1_codeql_by_strategy.png` | CodeQL Alerts by Strategy |
| 2 | `chart2_codeql_by_version.png` | CodeQL Alerts by Version |
| 3 | `chart3_dast_by_version.png` | DAST FAIL/WARN/PASS by Version |
| 4 | `chart4_dast_by_strategy.png` | DAST Avg WARN/PASS & Server Leak by Strategy |
| 5 | `chart5_dast_heatmap.png` | DAST Warning Types Heatmap |
| 6 | `chart6_sonarqube_by_strategy.png` | SonarQube Issues by Strategy |
| 7 | `chart7_sonarqube_heatmap.png` | SonarQube Issues Heatmap |
| 8 | `chart8_sonarqube_duplications.png` | SonarQube Duplications by Version |
| 9 | `chart9_unit_tests_heatmap.png` | Unit Test Pass Rate Heatmap |
| 10 | `chart10_radar_comparison.png` | Radar — Overall Strategy Comparison |